# BERTopic - topic modelling of news descriptions 

[BERTopic](https://maartengr.github.io/BERTopic/index.html) is a library for doing topic modelling with BERT language model. Maybe you have heard of an alternative - LDA topic modelling - this approach is based on different technical point of view than LDA.  BERTopic is based on dimensionality reduction and clustering of BERT sentence embeddings. More details of the algorithm you can find [here](https://maartengr.github.io/BERTopic/algorithm/algorithm.html#4-bag-of-words). 

First, we have to import some useful libraries in addition to BERTopic. We import pandas and numpy for processing the data. If you haven't installed these libraries already, you can check [install pandas](https://pandas.pydata.org/docs/getting_started/install.html) and [install numpy](https://numpy.org/install/) descriptions. 

In [ ]:
import pandas as pd 
import numpy as np
from bertopic import BERTopic

Our dataset is a news headline & abstract dataset from  [Kaggle](https://www.kaggle.com/datasets/rmisra/news-category-dataset). The dataset includes around 210 000 records, each record including multiple attributes such as authors, headline, link to the original article, and description of the news article. In our project, we are mainly interested in the descriptions. 

Next, we read our dataset with the suitable command. When we print the length of the dataframe, we see that it is quite big, almost 210000 items, as described above. We want to reduce the size of the dataset and we do that with indexes. 

In [ ]:
df_full = pd.read_json("News_Category_Dataset_v3.json", lines=True)
print(len(df_full))
df = df_full[0:1000]

In [ ]:
df_full.head()

In [ ]:

df_categories = pd.DataFrame(df_full.category.value_counts())
df_categories.head()
df_categories.to_csv('data/categories.csv')

## Training the model 

Finally we get into the training of the model. We define our BERTopic model as `model`, take a look to short descriptions by printing the first of them and then define a list of documents from the short descriptions. We start training the model with `fit_transform` function. 

In [ ]:
print(df['short_description'][0])
documents = df['short_description'].to_list()

model = BERTopic(verbose=True)

topics, probabilities = model.fit_transform(documents)

Let's take a look to the topics. We see, that the first topic, is not so understandable. It is mostly about pronouns and conjunctions. The one weird word among others is "dog". Another topic is clearly about war in Ukraine, we can see it from words such as "ukraine", "war" and "kyiv". 

In [ ]:
print(model.get_topic(0))
print("-----------------")
print(model.get_topic(1))

## Topic visualization

Next we visualize topics based on their size (number of documents belonging to a topic) and how near topics are each others based on the words of the topics. With sliding the bar, we can focus on specific topics. 

In [ ]:
model.visualize_topics()

## Topic similarity heatmap

We can also visualize how similar topics are to each others. This kind of heatmap is in general a common way to visualize data in Data Science applications. In the diagonal with blue color, we can see, that topics are fully similar (similarity score 1) with each others. Then we read the similarities by taking one item from x-axis and one item from y-axis. The similarity is calculated as cosine similarity of topic embeddings. 

In [ ]:
model.visualize_heatmap()

## Topics for new documents 

Now let's take some documents that the topic model hasn't seen yet. Let's take 100 more documents from the full dataset and get the probabilities of different topics for them with `fit_transform` function.  

In [ ]:
docs_test = df_full[1000:1050]['short_description'].to_list()

topics, probs = model.fit_transform(docs_test)
df_topics = pd.DataFrame({'topic': topics, 'document': docs_test})

df_topics.head()

In [ ]:
df_topics.topic.value_counts()

We can see that topics are represented as numbers. Now let's remind ourselves of how to take a look to topics and print a document and it's topic words. 

In [ ]:
for index, row in df_topics.iterrows():
    print(row["document"])
    print(model.get_topic(row["topic"]))

Let's make our printing a bit nicer by dropping the probabilities of topics from printing. If `zip` function isn't familiar for you, you can read from it [here](https://www.programiz.com/python-programming/methods/built-in/zip), from the example 3. So, in our case, `zip` function unpacks the tuples into two lists, where the first list includes topic words. 

Now we have a nicer printing for documents and their topic words. 

In [ ]:
for index, row in df_topics.iterrows():
    print(row["document"])
    print("Topic words ", list(zip(*model.get_topic(row["topic"])))[0])
    print("---------------------------------------")

## Specify number of topics 

Number of wanted topics can be also specified when defining the model. The algorithm then combines similar topics with each others. Let's specify the number of wanted topics and then run fitting the model to our documents again. We can see that the topics are quite similar than when running the model without specifying the number of topics. It is though an unsolved problem, how many topics should be used in topic modelling. 

In [ ]:
model = BERTopic(nr_topics=10, verbose=True, calculate_probabilities=True) 
topics, probabilities = model.fit_transform(documents)

model.visualize_topics()

## Visualize topic probabilities for individual document 

Sometimes we are interested in digging deeper into individual documents. We can do it with `visualize_distribution` function. The function visualizes topic distribution for one document. Let's do it for the first document. 

In [ ]:
print(documents[0])
model.visualize_distribution(probabilities[0])

TODO: Let's pick a particular news category, for example technology, and train a model usign the dataset. Explore the topics for the particular category and come up with an interesting insight that would benefit our organization. Students learning data science methods in the AI age, for example.